In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
root_dir = "/content/drive/MyDrive/dataset"
timeseries_dir = f"{root_dir}/timeseries"
csv_dir = f"{root_dir}/csv"

In [ ]:
out_x_path = f"{csv_dir}/X_windows_full_ds.npy"
out_d_path = f"{csv_dir}/Y_disease_labels.npy"
out_a_path = f"{csv_dir}/Y_activity_labels.npy"

In [ ]:
import os


print("Timeseries_dir:", timeseries_dir)
print("exists?", os.path.exists(timeseries_dir))
if os.path.exists(timeseries_dir):
    name=sorted(os.listdir(timeseries_dir))[:10]
    print("Samples files :", name)
print("\ncsv_dir:", csv_dir, "| Exists?", os.path.exists(csv_dir))

Timeseries_dir: /content/drive/MyDrive/dataset/timeseries
exists? True
Samples files : ['001_CrossArms_LeftWrist.txt', '001_CrossArms_RightWrist.txt', '001_DrinkGlas_LeftWrist.txt', '001_DrinkGlas_RightWrist.txt', '001_Entrainment_LeftWrist.txt', '001_Entrainment_RightWrist.txt', '001_HoldWeight_LeftWrist.txt', '001_HoldWeight_RightWrist.txt', '001_LiftHold_LeftWrist.txt', '001_LiftHold_RightWrist.txt']

csv_dir: /content/drive/MyDrive/dataset/csv | Exists? True


In [ ]:
df = pd.read_csv(f"{csv_dir}/file_list.csv")
df['id'] = df['id'].astype(int).astype(str).str.zfill(3)
df.head()

,resource_type,id,study_id,condition,disease_comment,age_at_diagnosis,age,height,weight,gender,handedness,appearance_in_kinship,appearance_in_first_grade_kinship,effect_of_alcohol_on_tremor,label
0,patient,001,PADS,Healthy,-,56,56,173,78,male,right,True,True,Unknown,0
1,patient,002,PADS,Other Movement Disorders,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,right,False,NaN,No effect,2
2,patient,003,PADS,Healthy,-,45,45,170,78,female,right,False,NaN,Unknown,0
3,patient,004,PADS,Parkinson's,IPS akinetic-rigid type,63,67,161,90,female,right,False,NaN,No effect,1
4,patient,005,PADS,Parkinson's,IPS tremordominant type,65,75,172,86,male,left,False,NaN,Unknown,1


In [ ]:
label_names = {0:'Healthy', 2:'DD', 1:'Parkinson'}
df['label_name'] = df['label'].map(label_names)
labeled_df =  df[['id','label']]
disease_map ={'Parkinson':1, 'DD':2, 'Healthy':0}
activities = [
    'CrossArms', 'DrinkGlas','Entrainment','HoldWeight','LiftHold', 'PointFinger','Relaxed','StretchHold','TouchIndex','TouchNose'
]
activity_map = {a:i for i,a in enumerate(activities)}
activity_df = pd.DataFrame(list(activity_map.items()), columns=['activity', 'activity_label'])
print("Subjects", len(labeled_df))
print(labeled_df.columns)
activity_df.head(10)

Subjects 469
Index(['id', 'label'], dtype='object')


,activity,activity_label
0,CrossArms,0
1,DrinkGlas,1
2,Entrainment,2
3,HoldWeight,3
4,LiftHold,4
5,PointFinger,5
6,Relaxed,6
7,StretchHold,7
8,TouchIndex,8
9,TouchNose,9


In [ ]:
sampling_rate = 100
window_size_sec = 1
overlap_percent = 10
window_size = int(window_size_sec*sampling_rate)
step_size = max(1, int(window_size*(1-overlap_percent/100)))
print('Window Size:', window_size)
print('Step Size:', step_size)

Window Size: 100
Step Size: 90


In [ ]:
def read_timeseries_txt(path):
  cols = ['Time', 'Acc_x','Acc_y','Acc_z','Gyro_x','Gyro_y','Gyro_z']
  df = pd.read_csv(path, sep=',',header=None, engine='python')
  if df.shape[1]!=7:
    raise ValueError(f"Unexpected column count in {path}:got {df.shape[1]}")
  df.columns = cols
  df_f= df.drop(columns=['Time']).astype('float32').values
  return df_f

In [ ]:
def extract_window_batched(left_file, right_file, disease_label, activity_label, batch_size=256):
  left_features = read_timeseries_txt(left_file)
  right_features = read_timeseries_txt(right_file)

  n_rows = min(len(left_features), len(right_features))
  if n_rows<window_size:
    return
  windows, d_labels, a_labels = [],[],[]
  for start in range(0, n_rows-window_size+1, step_size):
    end = start+window_size
    window = np.concatenate([left_features[start:end], right_features[start:end]], axis=1)
    windows.append(window)
    d_labels.append(disease_label)
    a_labels.append(activity_label)

    if len(windows)>= batch_size:
      yield np.stack(windows), np.array(d_labels, dtype=np.int16),np.array(a_labels, dtype=np.int16)
      windows, d_labels, a_labels = [],[],[]

    if windows:
      yield np.stack(windows), np.array(d_labels, dtype=np.int16),np.array(a_labels, dtype=np.int16)

In [ ]:
# Pick the first subject id from labeled_df
test_id = labeled_df['id'].iloc[0]   # e.g. "001"
test_activity = 'StretchHold'

# Build file paths
left_test = os.path.join(timeseries_dir, f"{test_id}_{test_activity}_LeftWrist.txt")
right_test = os.path.join(timeseries_dir, f"{test_id}_{test_activity}_RightWrist.txt")

print("Testing files:")
print("Left:", left_test, "|exists:", os.path.exists(left_test))
print("Right:", right_test, "|exists:", os.path.exists(right_test))

# Run only if files exist
if os.path.exists(left_test) and os.path.exists(right_test):
    subject_label = labeled_df.loc[labeled_df['id'] == test_id, 'label'].values[0]
    dlab = subject_label
    alab = activity_map[test_activity]

    for xb, db, ab in extract_window_batched(left_test, right_test, dlab, alab, batch_size=64):
        print("Test batch shape X:", xb.shape, "Y Disease:", db, "Y Activity:", ab)
        break
else:
    print("Skipping testing")


Testing files:
Left: /content/drive/MyDrive/dataset/timeseries/001_StretchHold_LeftWrist.txt |exists: True
Right: /content/drive/MyDrive/dataset/timeseries/001_StretchHold_RightWrist.txt |exists: True
Test batch shape X: (1, 100, 12) Y Disease: [0] Y Activity: [7]


In [ ]:
X_list, y_list_d, y_list_a = [],[],[]
for subject_id, disease_label in zip(labeled_df['id'], labeled_df['label']):
  for act in activities:
    left_file = os.path.join(timeseries_dir, f"{subject_id}_{act}_LeftWrist.txt")
    right_file = os.path.join(timeseries_dir, f"{subject_id}_{act}_RightWrist.txt")

    if os.path.exists(left_file) and os.path.exists(right_file):
      activity_label = activity_map[act]

      for xb, db, ab in extract_window_batched(left_file, right_file, disease_label, activity_label, batch_size=256):
        X_list.append(xb)
        y_list_d.append(db)
        y_list_a.append(ab)

    else:
      print(f"Skipping {subject_id} {act}")
X_final = np.vstack(X_list)
y_final_d = np.concatenate(y_list_d)
y_final_a = np.concatenate(y_list_a)

print("Final dataset shape:")
print("X:", X_final.shape, "Y Disease:", y_final_d.shape, "Y Activity:", y_final_a.shape)

np.save(out_x_path, X_final)
np.save(out_d_path, y_final_d)
np.save(out_a_path, y_final_a)

print("Full dataset windowed and saved")

Final dataset shape:
X: (484946, 100, 12) Y Disease: (484946,) Y Activity: (484946,)
Full dataset windowed and saved


In [ ]:

csv_dir = "/content/drive/MyDrive/dataset/csv"
out_x_path = f"{csv_dir}/X_windows_full_ds.npy"
out_d_path = f"{csv_dir}/Y_disease_labels.npy"
out_a_path = f"{csv_dir}/Y_activity_labels.npy"

X_final = np.load(out_x_path)
y_final_d = np.load(out_d_path)
y_final_a = np.load(out_a_path)

print("Loaded dataset shapes:")
print("X:", X_final.shape, "Y Disease:", y_final_d.shape, "Y Activity:", y_final_a.shape)

X_flat = X_final.reshape(X_final.shape[0], -1)

combined = np.column_stack((y_final_d, y_final_a, X_flat))

combined_out_path = f"{csv_dir}/full_dataset_with_labels.txt"
np.savetxt(combined_out_path, combined, delimiter=",", fmt="%.5f")

print(" Combined TXT file saved at:", combined_out_path)
print("Format per row: [disease_label, activity_label, f1, f2, ..., f1200]")



Loaded dataset shapes:
X: (484946, 100, 12) Y Disease: (484946,) Y Activity: (484946,)
 Combined TXT file saved at: /content/drive/MyDrive/dataset/csv/full_dataset_with_labels.txt
Format per row: [disease_label, activity_label, f1, f2, ..., f1200]
